# Exploring a capture

Reads a recorded session and the tick table derived from it. Nothing here
depends on `book/`, `signals/` or `tca/`, so it runs against the repository as
it stands.

```bash
uv run l2tca synth --updates 20000 --out data/raw/sample.jsonl
uv run l2tca convert data/raw/sample.jsonl --out data/parquet
```

Synthetic data has no market meaning — swap in a real capture before drawing
any conclusion.

In [ ]:
from pathlib import Path

import polars as pl

from l2tca.feed.replay import iter_raw_messages, read_header
from l2tca.io import read_table

CAPTURE = Path("../data/raw/sample.jsonl")
PARQUET = Path("../data/parquet")

## The capture header

Pairs the monotonic clock with wall clock, so the file's timestamps can be placed on a calendar after the fact.

In [ ]:
header = read_header(CAPTURE)
header

## Inter-arrival times

The shape of this distribution is what the staleness watchdog is tuned against.

In [ ]:
from itertools import pairwise

recv = [m.recv_ns for m in iter_raw_messages(CAPTURE)]
gaps = pl.Series("gap_ms", [(b - a) / 1e6 for a, b in pairwise(recv)])
gaps.describe()

## The tick table

One row per price level touched, hour-partitioned.

In [ ]:
ticks = read_table(PARQUET, "tick")
ticks.head()

In [ ]:
ticks.group_by("side", "frame_type").agg(
    pl.len().alias("rows"),
    pl.col("is_delete").sum().alias("deletes"),
).sort("side", "frame_type")

## Touch prices over the session

The best bid and offer from the tick stream alone — a sanity check on the capture before any book code exists.

In [ ]:
touch = (
    ticks.filter(~pl.col("is_delete"))
    .group_by("seq", "side")
    .agg(
        pl.col("recv_ns").first(),
        pl.when(pl.col("side").first() == "bid")
        .then(pl.col("price").max())
        .otherwise(pl.col("price").min())
        .alias("touch"),
    )
    .sort("recv_ns")
)
touch.head(10)